# Hospital Readmission Prediction — Feature Engineering

## Overview
This notebook transforms the raw dataset into a clean, model-ready feature set.

Steps covered:
1. Handling missing values
2. Dropping uninformative columns  
3. Converting categorical age ranges to numeric
4. Engineering new predictive features
5. Encoding categorical variables
6. Saving the final cleaned dataset
---

In [1]:
import pandas as pd
import numpy as np

# Load original data again
df = pd.read_csv(r'C:\Users\munta\Documents\hospital-readmission-project\data\diabetic_data.csv')  # use your same path as before

# Replace ? with NaN
df.replace('?', np.nan, inplace=True)

# Drop columns with too much missing data
cols_to_drop = ['weight', 'payer_code', 'medical_specialty']
df.drop(cols_to_drop, axis=1, inplace=True)

# Drop ID columns - no predictive value
df.drop(['encounter_id', 'patient_nbr'], axis=1, inplace=True)

# Drop original readmitted and create binary target
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)
df.drop('readmitted', axis=1, inplace=True)

print("Shape after dropping columns:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape after dropping columns: (101766, 45)

Remaining nulls:
race              2273
diag_1              21
diag_2             358
diag_3            1423
max_glu_serum    96420
A1Cresult        84748
dtype: int64


## 1. Data Cleaning

### Missing Value Strategy
Based on EDA findings:
- `weight` (96.9% missing), `medical_specialty` (49.1%), `payer_code` (39.6%), 
  `max_glu_serum` (94.7%), `A1Cresult` (83.3%) → **Drop** (too much missing data)
- `race`, `diag_1/2/3` → **Impute** with mode (< 3% missing)
- ID columns `encounter_id`, `patient_nbr` → **Drop** (no predictive value)

In [2]:
# Drop high-missing columns we missed
df.drop(['max_glu_serum', 'A1Cresult'], axis=1, inplace=True)

# Convert age ranges to numeric midpoints
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

# Fill small missing values with mode
for col in ['race', 'diag_1', 'diag_2', 'diag_3']:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Nulls remaining:", df.isnull().sum().sum())
print("Shape:", df.shape)

Nulls remaining: 0
Shape: (101766, 43)


C:\Users\munta\AppData\Local\Temp\ipykernel_18864\4254651924.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


## 2. Age Encoding
Age is stored as bracket ranges (e.g. `[40-50)`). We convert these to numeric 
midpoint values to make them usable in ML models.

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

binary_cols = ['gender', 'change', 'diabetesMed']
for col in binary_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df = pd.get_dummies(df, columns=['race', 'insulin', 'metformin',
                                   'repaglinide', 'nateglinide',
                                   'glimepiride', 'glipizide',
                                   'glyburide', 'pioglitazone',
                                   'rosiglitazone', 'acarbose'],
                    drop_first=True)

df.drop(['diag_1', 'diag_2', 'diag_3'], axis=1, inplace=True)

# Save cleaned data
df.to_csv(r'C:\Users\munta\Documents\hospital-readmission-project\data\cleaned_data.csv', index=False)

print("Final shape:", df.shape)
print("Any nulls remaining:", df.isnull().sum().sum())
print("Saved to cleaned_data.csv")

Final shape: (101766, 63)
Any nulls remaining: 0
Saved to cleaned_data.csv


## 3. Feature Engineering
Creating new features that capture relationships not present in the raw columns.
These derived features represent domain knowledge about patient complexity.

In [4]:
# Create engineered features
df['meds_per_day'] = df['num_medications'] / df['time_in_hospital']
df['procedures_per_day'] = df['num_procedures'] / df['time_in_hospital']
df['total_prior_visits'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']
df['lab_per_day'] = df['num_lab_procedures'] / df['time_in_hospital']

print("New features created. Shape:", df.shape)
df[['meds_per_day', 'procedures_per_day', 'total_prior_visits', 'lab_per_day']].describe()

New features created. Shape: (101766, 67)


,meds_per_day,procedures_per_day,total_prior_visits,lab_per_day
count,101766.000000,101766.000000,101766.000000,101766.000000
mean,5.074465,0.452709,1.202759,14.381453
std,3.826038,0.872380,2.291781,12.060457
min,0.083333,0.000000,0.000000,0.071429
25%,2.600000,0.000000,0.000000,6.500000
50%,4.000000,0.142857,0.000000,11.000000
75%,6.166667,0.500000,2.000000,18.333333
max,42.000000,6.000000,80.000000,111.000000


### Engineered Features Rationale
| Feature | Formula | Clinical Rationale |
|---|---|---|
| `meds_per_day` | medications / stay length | Captures medication complexity relative to stay |
| `procedures_per_day` | procedures / stay length | Normalises procedural burden by stay duration |
| `total_prior_visits` | outpatient + emergency + inpatient | Overall prior healthcare utilisation |
| `lab_per_day` | lab procedures / stay length | Intensity of diagnostic workup |

## 4. Categorical Encoding
Converting categorical variables to numeric format required for ML models.
- **Label Encoding** for binary columns (2 categories)
- **One-Hot Encoding** for multi-category columns (creates dummy variables)

In [7]:
# Encoding already happened - just drop remaining unused columns
cols_to_drop = ['chlorpropamide', 'acetohexamide', 'tolbutamide', 
                'miglitol', 'troglitazone', 'tolazamide', 'examide', 
                'citoglipton', 'glyburide-metformin', 'glipizide-metformin',
                'glimepiride-pioglitazone', 'metformin-rosiglitazone', 
                'metformin-pioglitazone']

# Only drop columns that actually exist
cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df.drop(cols_to_drop, axis=1, inplace=True)

# Save final cleaned data
df.to_csv(r'C:\Users\munta\Documents\hospital-readmission-project\data\cleaned_data.csv', index=False)

print("Final shape:", df.shape)
print("Any nulls remaining:", df.isnull().sum().sum())
print("Saved successfully!")
print("\nFinal columns:", df.columns.tolist())

Final shape: (101766, 54)
Any nulls remaining: 0
Saved successfully!

Final columns: ['gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'change', 'diabetesMed', 'readmitted_binary', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'insulin_No', 'insulin_Steady', 'insulin_Up', 'metformin_No', 'metformin_Steady', 'metformin_Up', 'repaglinide_No', 'repaglinide_Steady', 'repaglinide_Up', 'nateglinide_No', 'nateglinide_Steady', 'nateglinide_Up', 'glimepiride_No', 'glimepiride_Steady', 'glimepiride_Up', 'glipizide_No', 'glipizide_Steady', 'glipizide_Up', 'glyburide_No', 'glyburide_Steady', 'glyburide_Up', 'pioglitazone_No', 'pioglitazone_Steady', 'pioglitazone_Up', 'rosiglitazone_No', 'rosiglitazone_Steady', 'rosiglitazone_Up', 'acarbose_No', 'acarbose_Steady', 'acarbose_Up', 'meds_

## Summary
- Started with **101,766 rows × 50 columns**
- Final dataset: **101,766 rows × 54 columns**
- Zero missing values remaining
- 4 new engineered features added
- All categorical variables encoded
- Saved to `data/cleaned_data.csv`